# Model Training and Evaluation
In this notebook, we will train several classification models, evaluate their performance, compare them, and save the best one for deployment.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'xgboost'

## 1. Load Data and Apply Preprocessing
We repeat the preprocessing steps (imputing, capping, dropping leakage columns, splitting, and scaling) to prepare the data for training.

In [ ]:
# Load Dataset
df = pd.read_csv('../dataset/Final_Flood_Project_Dataset.csv')

# Impute missing values with median
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Cap outliers for continuous variables
def cap_outliers(data, col):
    lower_percentile = data[col].quantile(0.05)
    upper_percentile = data[col].quantile(0.95)
    data[col] = np.where(data[col] < lower_percentile, lower_percentile, data[col])
    data[col] = np.where(data[col] > upper_percentile, upper_percentile, data[col])
    return data

for col in ['Annual_Rainfall', 'Seasonal_Rainfall']:
    if col in df.columns:
        df = cap_outliers(df, col)

# Train-Test Split (Dropping 'Flood_Probability' to prevent data leakage)
X = df.drop(['Flood', 'Flood_Probability'], axis=1)
y = df['Flood']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 2. Initialize Models
We initialize the four required algorithms: Decision Tree, Random Forest, K-Nearest Neighbors (KNN), and XGBoost.

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

## 3. Train and Evaluate Models
We train each model and calculate Accuracy, Precision, Recall, and F1 Score. We also generate the Confusion Matrix and Classification Report.

In [ ]:
results = []

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

for idx, (name, model) in enumerate(models.items()):
    print(f"\n--- Training and Evaluating: {name} ---")
    
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'Model Object': model
    })
    
    # Print Classification Report
    print(f"Classification Report for {name}:")
    print(classification_report(y_test, y_pred))
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx])
    axes[idx].set_title(f'Confusion Matrix: {name}')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.show()

## 4. ROC Curve Comparison
We plot the Receiver Operating Characteristic (ROC) curve for all models to visualize the trade-off between the True Positive Rate and False Positive Rate.

In [ ]:
plt.figure(figsize=(10, 8))

for name, model in models.items():
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Chance')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curves')
plt.legend(loc="lower right")
plt.show()

## 5. Compare Models and Select Best Model
We generate a comparison table of all metrics and automatically select the best model based on Accuracy.

In [ ]:
# Create comparison table
results_df = pd.DataFrame(results).drop('Model Object', axis=1)
print("Model Comparison Table:")
display(results_df)

# Select best model based on Accuracy
best_model_info = max(results, key=lambda x: x['Accuracy'])
best_model = best_model_info['Model Object']
best_model_name = best_model_info['Model']

print(f"\n✅ Best Model Automatically Selected: {best_model_name} with Accuracy of {best_model_info['Accuracy']:.4f}")

## 6. Save the Best Model
The best model is saved using `joblib` so it can be loaded into our Flask web application for real-time predictions.

In [ ]:
# Save best model to the designated path
joblib.dump(best_model, '../models/floods.save')
print(f"Model '{best_model_name}' successfully saved to ../models/floods.save")